In [1]:
#!pip install langchain langchain-core langchain_community langchain_openai
#Using langchain for templates
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate

In [2]:
#If numpy 2.2.6 (which might show as warning/error on execution of prev cell)
#!pip uninstall numpy -y
#!pip install "numpy<2"

In [3]:
#In case transformers is 4.x as done earlier
#!pip install tokenizers==0.15.2
#!pip uninstall langchain-huggingface -y

In [4]:
#Using different models
from transformers import pipeline

In [5]:
#downgrade protobuf to avoid warnings,if needed
#!pip install --upgrade protobuf==4.25.3
#restart session,if above step done
#!pip show protobuf


In [ ]:
#Using smaller model
generator = pipeline("text2text-generation", model="google/flan-t5-base")

def get_completion(prompt):
    response = generator(prompt,max_new_tokens=100, do_sample=False)
    return (response[0]["generated_text"].strip())

print(get_completion("What is a DEFI in context of crypto world"))
#Using different variant i.e larger model, to run remove """ """

In [ ]:
#from transformers import pipeline
#Note**Access to model mistralai/Mistral-7B-Instruct-v0.1 is restricted. You must have access to it and be
#authenticated to access it. If yes, then we can use the code below.
#Note ** this will download large tensors,configs etc.. for this model, thus to run remove """ """ & then run
"""
generator = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.1")


def get_completion(prompt):
    # For instruction-tuned models, prepend with an instruction-style format
    instruction = f"<s>[INST] {prompt} [/INST]"
    response = generator(instruction, max_new_tokens=100, do_sample=False)
    return response[0]["generated_text"].split("[/INST]")[-1].strip()

print(get_completion("What is defi in context of crypto world?"))
"""


In [ ]:
##using Another heavier model
#Note ** this will download large tensors,configs etc.. for this model, thus to run remove """ """ & then run
"""
# Load a text-generation pipeline with an instruction-tuned model
generator = pipeline("text-generation", model="tiiuae/falcon-7b-instruct")

def get_completion(prompt):
    # For instruction-tuned models, prepend with an instruction-style format
    instruction = f"<s>[INST] {prompt} [/INST]"
    response = generator(instruction, max_new_tokens=100, do_sample=False)
    return response[0]["generated_text"].split("[/INST]")[-1].strip()

print(get_completion("What is defi in context of crypto world?"))
"""

In [ ]:
#Back to Chain Approach & using templates
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
# Load the model and tokenizer locally
model_name = "google/flan-t5-large"  # You can also use "google/flan-t5-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

client = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100, 
    torch_dtype=torch.float32,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

In [3]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
from langchain_community.llms import HuggingFacePipeline

template2 = " Please write a {length} review,of the book {book_title}. "
input_variables2 = [ "length", "book_title" ]
prompt = PromptTemplate(
    input_variables=input_variables2,
    template=template2
)
#To check
#formatted_prompt = prompt.format(length = "short", book_title = " House Of Dragon")
#print(formatted_prompt)
llm = HuggingFacePipeline(pipeline=client)

chain_new = prompt | llm
response = chain_new.invoke({
    "length": "short",
    "book_title": "House of Dragon"
})
print(response)


a spooky tale of dragons and dragon-like creatures


In [5]:
#Using function as defined above
formatted_prompt = prompt.format(length = "short", book_title = " House Of Dragon")
def get_completion(prompt):
    response = client(prompt,max_new_tokens=100, do_sample=False)
    return (response[0]["generated_text"].strip())
response = get_completion(formatted_prompt)
print("AI Response:")
print(type(response))
print(response)

AI Response:
<class 'str'>
a spooky tale of dragons and dragon-like creatures


### Switching to usage of bigger LLMs deployed on endpoints

In [ ]:
#Using OpenAI and gpt model
#Note** If using gpt model and AzureOpenAI or AzureChatOpenAI (refer: 'Working_with_AzureOpenAI' folder)
import openai
from openai import AzureOpenAI
# Initialize client once
from dotenv import load_dotenv
#load_dotenv("/content/.env")
load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version="2024-12-01-preview",
)

#or
'''
#Using Langchain Equivalent
from langchain_openai import AzureChatOpenAI

#from dotenv import load_dotenv
#load_dotenv("/content/.env")
#load_dotenv()

client = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("API_KEY"),
    api_version="2024-12-01-preview",
    deployment_name="gpt-4.1",
    temperature=0,
)

llm.invoke("Explain transformers in simple terms in 25 words").content
'''

In [ ]:
#Creating function
def get_completion(prompt, deployment_name="gpt-4.1-mini"):
    """Get a chat completion from Azure OpenAI.
    Args:
        prompt (str): User input prompt.
        deployment_name (str): The deployment name you gave your model in Azure portal.
    Returns:
        dict: Full response object, or error dict.
    """
    try:
        messages = [{"role": "user", "content": prompt}]
        response = client.chat.completions.create(
            model=deployment_name,    # <-- This is the "deployment name" not the raw model name
            messages=messages,
            temperature=0.1,
            top_p=0.8,
            max_tokens=512
        )
        #return response.model_dump()  # Return the full response as dict
        # Extract just the assistant's reply
        return response.choices[0].message.content
    except Exception as e:
        return {"error": str(e)}

In [ ]:
response = get_completion(prompt)
response

In [ ]:
#If using get_completion() based on gpt model as defined above, then we can
"""
response = get_completion(formatted_prompt)
print("AI Response:")
print(type(response))
print(response.keys())

response['choices'][0]['message']['content']"""

### Formatting

In [ ]:
#Jinja Template example
jinja2_template = "Give me an {{ adjective }} fact about {{ topic }}"

In [ ]:
prompt = PromptTemplate.from_template(jinja2_template, template_format = "jinja2" )

In [ ]:
user_question = prompt.format(adjective="interesting", topic="space exploration")
print(user_question)

Give me an interesting fact about space exploration


In [ ]:
response = get_completion(user_question)
print("AI Response:")
print(response)

AI Response:
Space exploration is a form of exploration of the universe.


In [ ]:
#Using f-string (example)
fstring_template = "Here is a brief summary for the book titled '{book_title}':"
book_title = "The Great Gatsby"
prompt = fstring_template.format(book_title=book_title)

In [ ]:
#Testing a dummy function
def get_book_summary(prompt):
    return "It's a novel about love, wealth, and aspiration, set in the Roaring '20s."

In [ ]:
summary = get_book_summary(prompt)
print(summary)

It's a novel about love, wealth, and aspiration, set in the Roaring '20s.


In [ ]:
#Using function that invokes the LLM
response = get_completion(prompt)
print("AI Response:")
print(response)

AI Response:
'The Great Gatsby' is a book about a young man's journey from a small town to a big city.


In [ ]:
# Example where string prompt template would not work
# Define the prompt template

jinja2_template = """

Dear {{ name }},
{% if age < 18 %}
You are invited to our kids' event with activities such as face painting, bouncy castles, and clown shows.
{% elif age < 65 %}
You are invited to our adult event with activities like live music, wine tasting, and art workshops.
{% else %}
You are invited to our senior event with activities including book clubs, chess tournaments, and tea dances.
{% endif %}
Sincerely,
Event Organizer

Write the mail in 200 words
"""

In [ ]:
prompt = PromptTemplate.from_template(jinja2_template, template_format="jinja2")

# Format the prompt with specific values for 'action', 'group', and 'time_period'
argument_prompt = prompt.format(name="John Doe", age=12)

In [ ]:
response = get_completion(argument_prompt)
print("AI Response:")
print(response)

AI Response:
Dear John Doe, You are invited to our kids' event with activities such as face painting, bouncy castles, and clown shows. Sincerely, Event Organizer


In [ ]:
#Using Langchain templates & chat mode

In [ ]:
simple_prompt = "The {subject} is strong in this one."
human_prompt = "Summarize our conversation so far in {word_count} words."

In [ ]:
from langchain_core.prompts import HumanMessagePromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [ ]:
simple_message_template = HumanMessagePromptTemplate.from_template(simple_prompt)
human_message_template = HumanMessagePromptTemplate.from_template(human_prompt)

chat_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="conversation"),
    simple_message_template,
    human_message_template
])

In [ ]:
human_message = HumanMessage(content="What's the best way to learn a new language?")
ai_message = AIMessage(content="""\
1. Immerse yourself in the language: Try to use the language in your daily life as much as possible.
2. Practice regularly: Consistency is key when learning a new language.
3. Use language learning apps: There are many apps that can help you learn a new language in a fun and engaging way.\
""")

In [ ]:
conversation = chat_prompt.format_prompt(
    conversation=[human_message, ai_message],
    subject="Force",
    word_count="10"
).to_messages()

In [ ]:
print(conversation)

[HumanMessage(content="What's the best way to learn a new language?", additional_kwargs={}, response_metadata={}), AIMessage(content='1. Immerse yourself in the language: Try to use the language in your daily life as much as possible.\n2. Practice regularly: Consistency is key when learning a new language.\n3. Use language learning apps: There are many apps that can help you learn a new language in a fun and engaging way.', additional_kwargs={}, response_metadata={}), HumanMessage(content='The Force is strong in this one.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Summarize our conversation so far in 10 words.', additional_kwargs={}, response_metadata={})]


In [ ]:
simple_prompt = "The {subject} is fascinating to study."
human_prompt = "Summarize our conversation so far in {word_count} words."

In [ ]:
human_message = HumanMessage(content="What's happens inside a black hole")
ai_message = AIMessage(content="""\
1. Inside black hole gariivty is zero way.\
""")

In [ ]:
conversation = chat_prompt.format_prompt(
    conversation=[human_message, ai_message],
    subject="Black Hole",
    word_count="10"
).to_messages()

In [ ]:
prompt_string = conversation
print("Formatted Prompt:")
print(prompt_string)


Formatted Prompt:
[HumanMessage(content="What's happens inside a black hole", additional_kwargs={}, response_metadata={}), AIMessage(content='1. Inside black hole gariivty is zero way.', additional_kwargs={}, response_metadata={}), HumanMessage(content='The Black Hole is strong in this one.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Summarize our conversation so far in 10 words.', additional_kwargs={}, response_metadata={})]


In [ ]:
print(type(prompt_string))

<class 'list'>


In [ ]:
#To be fixed in format
#response = get_completion(prompt_string)
#print(response)

In [ ]:
#Using Langchain templates & styles

In [ ]:
#Using style
#Modify parameters like **customer_style** and **customer_email**
#to influence the tone and formality of the generated responses.

template_string = """Translate the text that is delimited by triple backticks into a style
that is {style}. text: ```{text}```"""

# Style and email input
customer_style = "American English in a casual tone"
customer_email = """
I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!
"""

In [ ]:
prompt = template_string.format(style=customer_style, text=customer_email)

In [ ]:
instruction_prompt = f"<s>[INST] {prompt} [/INST]"

In [ ]:
#Using generator based on google/flan-t5-base
response = generator(instruction_prompt, max_new_tokens=150, do_sample=False)

In [ ]:
response

[{'generated_text': " I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!  [/INST]"}]

In [ ]:
generated_text = response[0]['generated_text'].split("[/INST]")[-1].strip()

In [ ]:
print(response[0])

{'generated_text': " I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!  [/INST]"}


In [ ]:
template_string = """Translate the text that is delimited by triple backticks into a style that is {style}.
text: ```{text}```"""

prompt_template = ChatPromptTemplate.from_template(template_string)

messages = prompt_template.format_messages(
    style="Scottish English in a professional tone",
    text="I'm super excited about the new gaming console & new game!"
)

#Using generator based on google/flan-t5-base
response = generator(prompt,max_new_tokens=150, do_sample=False)
print(response)

[{'generated_text': " I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!"}]
